# Agentic RAG 

**Naive RAG** dumps every source into one index, retrieves top-k, and answers. No decisions.

**Agentic RAG** lets the LLM *decide* about retrieval. The simplest, most legible decision is **routing**: given a question, the agent picks *which* source to search. This dataset is built for it — four distinct hiring corpora become four tools, and the agent is the router.

We build **one** `create_agent` (LangChain 1.0). No LangGraph, no hand-wired graph. Then we watch two things happen in the *same* agent:

1. **Routing** — a single-intent question → the agent calls exactly one tool.
2. **Decomposition** — a compound question → the agent splits it and calls several tools. Same agent, harder question.

We nudge the decomposition with a **prompt instruction** ("list the parts, then call one tool per part") — global planning without building a separate planner node.

## The four spokes

| Source | Tool | Contents |
|---|---|---|
| `leveling` | `search_leveling_docs` | Level rubrics, role expectations per IC level |
| `interview_process` | `search_interview_process` | Interview loops, rounds, scorecards |
| `comp_guide` | `search_comp_guide` | Salary bands, tier/YoE adjustments |
| `hiring_history` | `search_hiring_history` | Past offers, precedent |

![Diagram](./images/01_agentic_rag.png)

## Setup

In [5]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [6]:
# Load .env, overriding any pre-existing environment variables.
load_dotenv(override=True)

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your environment."

## 1. Build the four spokes

Each source gets its own Chroma collection. Separate indexes (not one mega-index) are what *make routing possible* — the agent can target one corpus instead of fishing in a blended one, and citations stay clean.

![Diagram](./images/02_retrievers.png)

In [7]:
SOURCES = {
    "leveling":          "data/leveling/",
    "interview_process": "data/interview_process/",
    "comp_guide":        "data/comp_guide/",
    "hiring_history":    "data/hiring_history/",
}

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def load_markdown_docs(folder: str) -> list[Document]:
    """Read every .md file under `folder` into a Document, tagging each with its
    path as `source` metadata (so citations like [data/comp_guide/...] still work).

    Plain pathlib + Document — no `langchain-community` loaders needed. That package
    is being sunset; for simple local files this is the lighter, future-proof path.
    """
    return [
        Document(page_content=path.read_text(encoding="utf-8"), metadata={"source": str(path)})
        for path in sorted(Path(folder).glob("**/*.md"))
    ]

def build_retriever(folder: str, k: int = 4):
    docs = load_markdown_docs(folder)
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=600, chunk_overlap=80
    ).split_documents(docs)
    store = Chroma.from_documents(
        chunks, embeddings,
        collection_name=folder.strip("/").replace("/", "_"),
    )
    return store.as_retriever(search_kwargs={"k": k})

RETRIEVERS = {name: build_retriever(path) for name, path in SOURCES.items()}
print(f"Built {len(RETRIEVERS)} retrievers: {list(RETRIEVERS.keys())}")

Built 4 retrievers: ['leveling', 'interview_process', 'comp_guide', 'hiring_history']


## 2. Wrap each spoke as a tool

**The docstring is the prompt.** It is the only thing the agent reads to decide which tool fits a sub-question. A vague docstring is a vague prompt — be specific about *what's inside* and *when to use it*.

![Diagram](./images/03_retriver_tools.png)

In [8]:
def format_docs(docs) -> str:
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('source','unknown')}]\n{d.page_content}" for d in docs
    )

@tool
def search_leveling_docs(query: str) -> str:
    """Search engineering leveling rubrics for role expectations, scope of impact,
    and competency requirements across IC levels (L3 Senior through L7 Principal).
    Use for questions about what a level means or what is expected at a given level."""
    return format_docs(RETRIEVERS["leveling"].invoke(query))

@tool
def search_interview_process(query: str) -> str:
    """Search interview loop templates and scorecards by role and level. Returns
    number and types of rounds, panel composition, bar-raiser policy, and
    team-specific adjustments. Use for questions about HOW we interview."""
    return format_docs(RETRIEVERS["interview_process"].invoke(query))

@tool
def search_comp_guide(query: str) -> str:
    """Search the compensation guide for salary bands, equity ranges, signing-bonus
    policy, prior-company tier adjustments (FAANG / Tier 2 / Tier 3), and
    years-of-experience heuristics. Use for any question about compensation."""
    return format_docs(RETRIEVERS["comp_guide"].invoke(query))

@tool
def search_hiring_history(query: str) -> str:
    """Search past offer memos and hiring debriefs for precedent — what we offered
    similar candidates, calibration decisions, lessons learned. Use when the
    question asks about precedent or 'what did we do last time'."""
    return format_docs(RETRIEVERS["hiring_history"].invoke(query))

RETRIEVAL_TOOLS = [
    search_leveling_docs,
    search_interview_process,
    search_comp_guide,
    search_hiring_history,
]

## 3. The agent (one `create_agent`)

This is the whole orchestration. The system prompt does the **Level-2 decomposition nudge**: *first list the sub-questions, then call one tool per sub-question.* That gives the agent **global** planning (commit to the full set of parts before retrieving) instead of lazy **greedy** planning (answer one part, declare done) — without a separate planner node.

![Diagram](./images/04_agent_loop.png)

In [9]:
SYSTEM = """You are a senior engineering recruiter answering hiring-manager questions, \
grounded ONLY in four searchable sources:
- search_leveling_docs        — level expectations / rubrics
- search_interview_process    — interview loop, rounds, scorecards
- search_comp_guide           — salary bands, tier & YoE adjustments
- search_hiring_history        — precedent from past offers

How to work:
1. DECOMPOSE: first break the question into its distinct parts (sub-questions). A
   3-part question has 3 parts. Write them out as a short numbered PLAN before
   acting, and for each part name the ONE tool you intend to call. Example:
   "PLAN: (1) interview rounds -> search_interview_process; (2) comp -> search_comp_guide".
2. ROUTE & RETRIEVE: for each part, call the ONE tool whose description best fits.
   A single-part question = one tool call. Don't call tools you don't need.
3. ANSWER: address every part. Use specific numbers/ranges from the evidence
   (say "$250-310k base", not "competitive"). Cite the source after each claim
   with a bracket tag: [leveling] [interview_process] [comp_guide] [hiring_history].
   If the evidence doesn't cover something, say so — do not guess.
"""

agent = create_agent(
    model="openai:gpt-4.1",
    tools=RETRIEVAL_TOOLS,
    system_prompt=SYSTEM,
)
print("Agent ready.")

Agent ready.


A tracer so we can *see* the agent's whole thought process, in order: the **PLAN** it writes (decomposition), the **tool calls** it makes (routing) with their search queries, the **evidence** each retrieval returns, and the final **answer**. This is where routing and decomposition stop being abstract and become a visible timeline. Pass `show_evidence=False` for a compact view.

In [10]:
import textwrap

def run(question: str, show_evidence: bool = True, max_chars: int = 400):
    """Run the agent and print its full thought process as a timeline:
    PLAN (decomposition) -> TOOL CALLS (routing) -> EVIDENCE (retrieval) -> ANSWER.
    """
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    msgs = result["messages"]

    print("=" * 72)
    print(f"Q: {question}")
    print("=" * 72)

    n_calls = 0
    for m in msgs[:-1]:  # last message is the final answer, printed separately below
        # The agent's own words: its decomposition PLAN / reasoning before acting.
        if m.type == "ai" and str(m.content).strip():
            print("\n🧠 PLAN / REASONING")
            print(textwrap.indent(str(m.content).strip(), "   "))

        # Routing decisions: which tool, with what search query.
        for tc in (getattr(m, "tool_calls", None) or []):
            n_calls += 1
            print(f"\n🔧 TOOL CALL #{n_calls}: {tc['name']}")
            print(f"     query = {tc['args'].get('query', '')!r}")

        # What each retrieval actually returned (the grounding evidence).
        if m.type == "tool" and show_evidence:
            snippet = str(m.content).strip()
            if len(snippet) > max_chars:
                snippet = snippet[:max_chars] + " …[truncated]"
            print(f"\n📄 EVIDENCE  <- {m.name}")
            print(textwrap.indent(snippet, "     "))

    print("\n" + "=" * 72)
    print(f"✅ ANSWER  ({n_calls} tool call(s) total)")
    print("=" * 72)
    print(msgs[-1].content)
    return result

## 4. Demo A — Routing (one intent → one tool)

A single-intent question. The agent should reach for **exactly one** source. Watch the trace: one tool call, and it's the right one.

In [11]:
_ = run("How many rounds are in the Staff Engineer interview loop, and what types?")

Q: How many rounds are in the Staff Engineer interview loop, and what types?

🔧 TOOL CALL #1: search_interview_process
     query = 'Staff Engineer interview loop number of rounds and types'

📄 EVIDENCE  <- search_interview_process
     [data/interview_process/staff_interview_loop.md]
     # Staff Engineer Interview Loop

     The Staff Engineer loop consists of 6 rounds, scheduled across 2-3 weeks for external candidates. Total candidate time investment: ~7 hours of interviews plus ~4-6 hours of preparation.

     ## The Six Rounds

     ### 1. Recruiter Screen (30 min)
     Conducted by a recruiter. Covers background, motivation, comp expectations (ear …[truncated]

✅ ANSWER  (1 tool call(s) total)
The Staff Engineer interview loop consists of 6 rounds:

1. Recruiter Screen (30 min) – Background, motivation, comp expectations, and basic fit.
2. Hiring Manager Screen (45 min) – Deep dive into candidate’s recent projects, scope, and reasoning.
3. Technical Deep Dive (60 min) – Walkthro

In [12]:
# Another single-intent question — routes to a different source.
_ = run("What's the comp band for a Staff Engineer?")

Q: What's the comp band for a Staff Engineer?

🔧 TOOL CALL #1: search_comp_guide
     query = 'Staff Engineer compensation band'

📄 EVIDENCE  <- search_comp_guide
     [data/comp_guide/comp_bands_2025.md]
     ## L5 Staff Detail

     For a Staff Engineer offer, the typical total comp (base + equity/4 + bonus) lands between $430,000 and $580,000 per year. The high end of the band is reserved for candidates with:
     - 8+ years of relevant experience
     - Strong prior signal from a FAANG or peer company (Tier 1)
     - Multiple competing offers (subject to comp committee approval)

     ## …[truncated]

✅ ANSWER  (1 tool call(s) total)
The compensation band for a Staff Engineer (L5) is:
- Base salary: $250,000 – $310,000
- Equity (4-year grant): $480,000 – $720,000 total value
- Target bonus: 15%

Typical total compensation (base + equity/4 + bonus) is $430,000 – $580,000 per year. The high end of the band is reserved for candidates with 8+ years of relevant experience, top-tier (FAA

## 5. Demo B — Decomposition emerges (compound question)

Same agent, no code changes. A four-part question. The decomposition nudge in the prompt makes the agent split it and **route each part** — the trace now shows several tool calls instead of one. Routing applied repeatedly *is* decomposition.

In [13]:
compound = (
    "ok so we're spinning up a staff eng req on the platform team and i'm fuzzy on "
    "a few things — what are we even expecting at staff level these days? and remind "
    "me what the loop looks like, how many rounds and what kind? oh and i've got a "
    "candidate coming from a faang shop, ~9 yrs in, where would comp land for them? "
    "trying to get this moving today, thanks"
)
_ = run(compound)

Q: ok so we're spinning up a staff eng req on the platform team and i'm fuzzy on a few things — what are we even expecting at staff level these days? and remind me what the loop looks like, how many rounds and what kind? oh and i've got a candidate coming from a faang shop, ~9 yrs in, where would comp land for them? trying to get this moving today, thanks

🧠 PLAN / REASONING
   PLAN:
   1. Staff-level expectations → search_leveling_docs
   2. Interview loop details for staff (platform) → search_interview_process
   3. Comp for FAANG candidate, ~9yrs exp, staff eng → search_comp_guide

   Calling all three tools in parallel.

🔧 TOOL CALL #1: search_leveling_docs
     query = 'staff engineer expectations'

🔧 TOOL CALL #2: search_interview_process
     query = 'staff engineer interview process platform team'

🔧 TOOL CALL #3: search_comp_guide
     query = 'staff engineer compensation faang 9 years experience'

📄 EVIDENCE  <- search_leveling_docs
     [data/leveling/staff_engineer_expectat

## Recap

- **Agentic RAG = the LLM decides about retrieval.** Here the decision is *which source* → routing.
- **One `create_agent` + N well-described tools** is the entire system. The docstrings are the routing logic.
- **Decomposition isn't a separate step** — it's routing applied to each part of a compound question, emerging from the same ReAct loop.
- **The Level-2 nudge** ("list the parts, then one tool per part" in the prompt) buys reliable global planning without building a planner node or touching LangGraph.

**Where you'd reach for more:** if you need a *strict* "is this evidence sufficient? retrieve again" check with a bounded retry, that's self-correction — the ReAct loop does it only loosely. That's the natural motivation for graduating to LangGraph in a follow-up.

> Note on citations: with a single agent, citation accuracy rests on the prompt asking the model to tag claims by source — there's no separate provenance-tracking step. Good enough for a tutorial; tighten it if you ship this.